# 01 — Pre-Metric EDA + Hypothesis Formulation

**Purpose:** Understand the distribution of retail ownership across the 20-stock universe,
visualize QoQ trends, spot-check 2–3 stocks visually, and write the formal H0/H1.

**Done condition:** Runs end-to-end; H0 and H1 written as markdown cells below.

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

DB_PATH = Path('../data/retail_behavior.db')
conn = sqlite3.connect(DB_PATH)
sns.set_theme(style='whitegrid')

## 1. Retail % Distribution Across Universe

In [ ]:
sh = pd.read_sql("""
    SELECT s.ticker, s.sector, s.expected_retail_tilt,
           AVG(sh.public_retail_pct) AS avg_retail_pct
    FROM stocks s
    JOIN shareholding sh ON s.ticker = sh.ticker
    GROUP BY s.ticker, s.sector
    ORDER BY s.sector, avg_retail_pct
""", conn)

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=False)
for ax, sector in zip(axes, ['Finance', 'Manufacturing']):
    sub = sh[sh['sector'] == sector]
    sns.barplot(data=sub, x='ticker', y='avg_retail_pct', ax=ax)
    ax.set_title(f'{sector} — Avg Retail %')
    ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

## 2. QoQ Retail % Trend Per Stock

In [ ]:
sh_qoq = pd.read_sql("""
    SELECT s.ticker, s.sector, sh.quarter_end_date, sh.public_retail_pct
    FROM stocks s
    JOIN shareholding sh ON s.ticker = sh.ticker
    ORDER BY s.sector, s.ticker, sh.quarter_end_date
""", conn)

# Spot-check: plot QoQ trend for 2-3 stocks
SPOT_CHECK = ['YESBANK', 'HDFCBANK', 'SUZLON']
fig, ax = plt.subplots(figsize=(10, 5))
for t in SPOT_CHECK:
    sub = sh_qoq[sh_qoq['ticker'] == t]
    ax.plot(sub['quarter_end_date'], sub['public_retail_pct'], marker='o', label=t)
ax.set_title('QoQ Retail % — Spot Check')
ax.legend()
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

## 3. Hypothesis Statement

**H0 (Null):** Stocks with high retail shareholding % show **no statistically significant difference**
in price/volume behavior around quarterly results dates compared to stocks with low retail
shareholding %, within the same sector.

**H1 (Alternative):** Stocks with high retail shareholding % show **significantly more erratic**
price/volume behavior (higher volume spike ratio, higher price range %, higher realized volatility)
around quarterly results dates compared to stocks with low retail shareholding %, within the same sector.

**Test:** Mann-Whitney U (two-sided), α = 0.05, within-sector median split bucketing.  
**Unit of observation:** one stock-quarter event window metric value.  
**Primary window:** ±3 trading days. **Robustness check:** ±5 trading days.